# Polynomial invariants for four qubits

In [1]:
import numpy as np
from src.mps_utils import get_product_state, get_ghz_state, to_comp_basis, get_rand_mps

In [ ]:
product_state = to_comp_basis(get_product_state(L=4, state_per_site=[0, 0, 0, 0]))
ghz_state = to_comp_basis(get_ghz_state(L=4))
W_state = get_W_state(L=4)
rmps_state = to_comp_basis(get_rand_mps(L=4, d_phys=2, chi=4))


def get_W_state(L: int = 4, dtype: np.dtype = np.float32):
    """
    Get the W state for a given number of qubits.
    """
    W_state = np.zeros(2**L, dtype=dtype)
    for i in range(L):
        W_state[2**i] = 1.0
    W_state = W_state / np.linalg.norm(W_state)
    return W_state

In [73]:
def compute_four_qubit_SLOCC_invariants(psi: np.ndarray):
    assert psi.shape == (16,)
    H = (psi[0] * psi[15] - psi[1] * psi[14] - psi[2] * psi[13] + psi[3] * psi[12] - psi[4] * psi[11] + psi[5] * psi[10] + psi[6] * psi[9] - psi[7] * psi[8]).item()

    L = np.linalg.det(psi.reshape(4, 4)).item()
    M = np.linalg.det([
        [psi[0], psi[8], psi[2], psi[10]],
        [psi[1], psi[9], psi[3], psi[11]],
        [psi[4], psi[12], psi[6], psi[14]],
        [psi[5], psi[13], psi[7], psi[15]],
    ]).item()
    Dxt = compute_D(psi, 0, 3).item()
    return abs(H), abs(L), abs(M), abs(Dxt)

rand_state = np.random.randn(16) + 1j * np.random.randn(16)
rand_state = rand_state / np.linalg.norm(rand_state)
compute_four_qubit_SLOCC_invariants(rand_state)

(0.21039171498983356,
 0.009693910522972223,
 0.007716032121713,
 0.0010023542244856128)

In [42]:
import sympy as sp

# basis monomials for degree-2 in two vars
def basis_deg2(x0, x1):
    return [x0**2, x0*x1, x1**2]

def compute_B_for_pair(A, u, v, symbolic=True):
    """
    Compute the 3x3 matrix B_{uv} for a 2x2x2x2 tensor A.
    A: numpy array shape (2,2,2,2) or nested list convertible to np.array
    u,v: two distinct indices chosen from {0,1,2,3} indicating which qubits are x and y.
    symbolic: if True uses sympy symbols for x,y; otherwise returns numeric B by
              choosing canonical basis vectors (not useful for symbolic D).
    Returns: sympy.Matrix 3x3 (if symbolic) or numpy.ndarray 3x3 (if not symbolic)
    """
    A = np.array(A)
    assert A.shape == (2,2,2,2)
    assert u != v and 0 <= u < 4 and 0 <= v < 4

    # remaining indices r,s (order matters only for how we form C)
    indices = [0,1,2,3]
    r, s = [i for i in indices if i not in (u, v)]

    # define symbolic variables for x and y
    x0, x1, y0, y1 = sp.symbols('x0 x1 y0 y1')
    bx = basis_deg2(x0, x1)
    by = basis_deg2(y0, y1)

    # build 2x2 matrix C_{k,l} = sum_{i,j} A[...,k,l] * x_i * y_j
    # We need to map positions of i,j to the tensor axes u,v and k,l to r,s.
    # We'll iterate over all indices of A to construct C.
    C = sp.Matrix([[0,0],[0,0]])
    for multi_idx in [(i0,i1,i2,i3) for i0 in (0,1) for i1 in (0,1) for i2 in (0,1) for i3 in (0,1)]:
        coeff = A[multi_idx]
        # pick components corresponding to u,v,r,s positions
        i = multi_idx[u]   # index for x
        j = multi_idx[v]   # index for y
        k = multi_idx[r]   # row in C
        l = multi_idx[s]   # col in C

        term = coeff * (x0 if i==0 else x1) * (y0 if j==0 else y1)
        # Add to C[k,l]
        C[k, l] = sp.simplify(C[k, l] + term)

    # compute determinant det(C) which is degree-2 in x and degree-2 in y
    detC = sp.expand(C.det())

    # collect coefficients into 3x3 matrix B such that detC = sum_{p,q} B[p,q]*bx[p]*by[q]
    # We'll solve for B entries by comparing coefficients of monomials bx[p]*by[q].
    # Create symbols for unknown B entries and set up linear equations.
    B_syms = sp.symbols('B0:9')
    Bmat = sp.Matrix(3, 3, B_syms)

    # form expression from Bsyms * basis monomials
    expr_from_B = sum(Bmat[i, j] * bx[i] * by[j] for i in range(3) for j in range(3))
    expr_from_B = sp.expand(expr_from_B)

    # create linear system by equating coefficients of all monomials in (x0,x1,y0,y1)
    # We'll collect terms by monomials of x and y separately; but easier: subtract and
    # equate coefficients of all independent monomials.
    diff = sp.expand(detC - expr_from_B)
    # get monomials present
    monoms = sp.Poly(diff, x0, x1, y0, y1).terms()
    # monoms is list of ((e_x0, e_x1, e_y0, e_y1), coeff)
    eqs = []
    # Extract coefficients for all degree-4 monomials (they will be 0)
    # But Sympy's Poly.terms gives only nonzero coefficients; we need to match coefficients for basis monomials
    # Build equations by equating coefficients of each basis term bx[i]*by[j].
    for i in range(3):
        for j in range(3):
            coeff_det = sp.expand(sp.Poly(detC, x0, x1, y0, y1).coeff_monomial(bx[i]*by[j]))
            # coefficient in expr_from_B for this monomial is Bmat[i,j]
            eqs.append(sp.Eq(Bmat[i,j], coeff_det))

    # Solve linear system (they're linear in Bsyms)
    sol = sp.solve(eqs, B_syms, dict=True)
    if not sol:
        raise RuntimeError("Could not solve for B matrix (unexpected).")
    sol = sol[0]
    # build numeric/symbolic B matrix
    B_result = sp.Matrix(3,3, [sp.simplify(sol[s]) for s in B_syms])
    return B_result

def D_for_pair(A, u, v):
    """
    Compute D_{uv} = det(B_{uv}) as a sympy expression.
    """
    B = compute_B_for_pair(A, u, v, symbolic=True)
    return sp.simplify(B.det())

# ---------------------------
# Example usage
# ---------------------------
# Example: A random integer 2x2x2x2 tensor (you can put your A entries here)
# order of axes is (q0, q1, q2, q3). Use whichever ordering you prefer

# choose a pair u,v
u, v = 2, 3  # compute D_{01} for qubits 0 and 1
state = rmps_state.reshape(2, 2, 2, 2)

import time
t0 = time.time()
D01 = D_for_pair(state, u, v)
t1 = time.time()
print(f"Time taken: {t1 - t0} seconds")
# print("A (flattened):", A.flatten())
# print(f"D_{{{u}{v}}} =")
# sp.pprint(D01)
D_new = compute_D(rmps_state, 0, 1)
H, L, M = compute_four_qubit_SLOCC_invariants(rmps_state)
print(H * M)
print(D_for_pair(state, 0, 3) - D_for_pair(state, 0, 1))
print(compute_D(rmps_state, 0, 3) - compute_D(rmps_state, 0, 1))

t2 = time.time()
print(compute_D(rmps_state, 0, 1))
print(compute_D(rmps_state, 2, 3))
print(compute_D(rmps_state, 0, 2))
print(compute_D(rmps_state, 1, 3))
print(compute_D(rmps_state, 0, 3))
print(compute_D(rmps_state, 1, 2))
t3 = time.time()
print(f"Time taken: {t3 - t2} seconds")


Time taken: 0.4403719902038574 seconds
-0.00046692289726311606
-0.000466922882224726
-0.0004669229
0.00046516757
0.00046516754
6.43379e-07
6.4339156e-07
-1.75535e-06
-1.7553614e-06
Time taken: 0.0002944469451904297 seconds


### Helper functions

In [37]:
# functions for the computation of D invariants

def compute_Dxy(a: np.ndarray):
    r0c0 = -a[1]*a[2] + a[0]*a[3]
    r0c1 = a[3]*a[4] - a[2]*a[5] - a[1]*a[6] + a[0]*a[7]
    r0c2 = -a[5]*a[6] + a[4]*a[7]
    
    r1c0 = a[3]*a[8] - a[2]*a[9] - a[1]*a[10] + a[0]*a[11]
    r1c1 = (a[7]*a[8] - a[6]*a[9] - a[5]*a[10] + a[4]*a[11] + 
            a[3]*a[12] - a[2]*a[13] - a[1]*a[14] + a[0]*a[15])
    r1c2 = a[7]*a[12] - a[6]*a[13] - a[5]*a[14] + a[4]*a[15]
    
    r2c0 = -a[9]*a[10] + a[8]*a[11]
    r2c1 = a[11]*a[12] - a[10]*a[13] - a[9]*a[14] + a[8]*a[15]
    r2c2 = -a[13]*a[14] + a[12]*a[15]

    return np.array([
        [r0c0, r0c1, r0c2],
        [r1c0, r1c1, r1c2],
        [r2c0, r2c1, r2c2]
    ])

def compute_Dxz(a: np.ndarray):
    r0c0 = -a[1]*a[4] + a[0]*a[5]
    r0c1 = -a[3]*a[4] + a[2]*a[5] - a[1]*a[6] + a[0]*a[7]
    r0c2 = -a[3]*a[6] + a[2]*a[7]
    
    r1c0 = a[5]*a[8] - a[4]*a[9] - a[1]*a[12] + a[0]*a[13]
    r1c1 = (a[7]*a[8] - a[6]*a[9] + a[5]*a[10] - a[4]*a[11] - 
            a[3]*a[12] + a[2]*a[13] - a[1]*a[14] + a[0]*a[15])
    r1c2 = a[7]*a[10] - a[6]*a[11] - a[3]*a[14] + a[2]*a[15]
    
    r2c0 = -a[9]*a[12] + a[8]*a[13]
    r2c1 = -a[11]*a[12] + a[10]*a[13] - a[9]*a[14] + a[8]*a[15]
    r2c2 = -a[11]*a[14] + a[10]*a[15]

    return np.array([
        [r0c0, r0c1, r0c2],
        [r1c0, r1c1, r1c2],
        [r2c0, r2c1, r2c2]
    ])

def compute_Dxt(a):
    r0c0 = -a[2]*a[4] + a[0]*a[6]
    r0c1 = -a[3]*a[4] - a[2]*a[5] + a[1]*a[6] + a[0]*a[7]
    r0c2 = -a[3]*a[5] + a[1]*a[7]
    
    r1c0 = a[6]*a[8] - a[4]*a[10] - a[2]*a[12] + a[0]*a[14]
    r1c1 = (a[7]*a[8] + a[6]*a[9] - a[5]*a[10] - a[4]*a[11] - 
            a[3]*a[12] - a[2]*a[13] + a[1]*a[14] + a[0]*a[15])
    r1c2 = a[7]*a[9] - a[5]*a[11] - a[3]*a[13] + a[1]*a[15]
    
    r2c0 = -a[10]*a[12] + a[8]*a[14]
    r2c1 = -a[11]*a[12] - a[10]*a[13] + a[9]*a[14] + a[8]*a[15]
    r2c2 = -a[11]*a[13] + a[9]*a[15]

    return np.array([
        [r0c0, r0c1, r0c2],
        [r1c0, r1c1, r1c2],
        [r2c0, r2c1, r2c2]
    ])

def compute_Dyz(a):
    r0c0 = -a[1]*a[8] + a[0]*a[9]
    r0c1 = -a[3]*a[8] + a[2]*a[9] - a[1]*a[10] + a[0]*a[11]
    r0c2 = -a[3]*a[10] + a[2]*a[11]
    
    r1c0 = -a[5]*a[8] + a[4]*a[9] - a[1]*a[12] + a[0]*a[13]
    r1c1 = (-a[7]*a[8] + a[6]*a[9] - a[5]*a[10] + a[4]*a[11] - 
            a[3]*a[12] + a[2]*a[13] - a[1]*a[14] + a[0]*a[15])
    r1c2 = -a[7]*a[10] + a[6]*a[11] - a[3]*a[14] + a[2]*a[15]
    
    r2c0 = -a[5]*a[12] + a[4]*a[13]
    r2c1 = -a[7]*a[12] + a[6]*a[13] - a[5]*a[14] + a[4]*a[15]
    r2c2 = -a[7]*a[14] + a[6]*a[15]

    return np.array([
        [r0c0, r0c1, r0c2],
        [r1c0, r1c1, r1c2],
        [r2c0, r2c1, r2c2]
    ])

def compute_Dyt(a):
    r0c0 = -a[2]*a[8] + a[0]*a[10]
    r0c1 = -a[3]*a[8] - a[2]*a[9] + a[1]*a[10] + a[0]*a[11]
    r0c2 = -a[3]*a[9] + a[1]*a[11]
    
    r1c0 = -a[6]*a[8] + a[4]*a[10] - a[2]*a[12] + a[0]*a[14]
    r1c1 = (-a[7]*a[8] - a[6]*a[9] + a[5]*a[10] + a[4]*a[11] - 
            a[3]*a[12] - a[2]*a[13] + a[1]*a[14] + a[0]*a[15])
    r1c2 = -a[7]*a[9] + a[5]*a[11] - a[3]*a[13] + a[1]*a[15]
    
    r2c0 = -a[6]*a[12] + a[4]*a[14]
    r2c1 = -a[7]*a[12] - a[6]*a[13] + a[5]*a[14] + a[4]*a[15]
    r2c2 = -a[7]*a[13] + a[5]*a[15]

    return np.array([
        [r0c0, r0c1, r0c2],
        [r1c0, r1c1, r1c2],
        [r2c0, r2c1, r2c2]
    ])

def compute_Dzt(a):
    r0c0 = -a[4]*a[8] + a[0]*a[12]
    r0c1 = -a[5]*a[8] - a[4]*a[9] + a[1]*a[12] + a[0]*a[13]
    r0c2 = -a[5]*a[9] + a[1]*a[13]
    
    r1c0 = -a[6]*a[8] - a[4]*a[10] + a[2]*a[12] + a[0]*a[14]
    r1c1 = (-a[7]*a[8] - a[6]*a[9] - a[5]*a[10] - a[4]*a[11] + 
            a[3]*a[12] + a[2]*a[13] + a[1]*a[14] + a[0]*a[15])
    r1c2 = -a[7]*a[9] - a[5]*a[11] + a[3]*a[13] + a[1]*a[15]
    
    r2c0 = -a[6]*a[10] + a[2]*a[14]
    r2c1 = -a[7]*a[10] - a[6]*a[11] + a[3]*a[14] + a[2]*a[15]
    r2c2 = -a[7]*a[11] + a[3]*a[15]

    return np.array([
        [r0c0, r0c1, r0c2],
        [r1c0, r1c1, r1c2],
        [r2c0, r2c1, r2c2]
    ])


def compute_D(psi: np.ndarray, i: int, j: int, return_det: bool = True):
    """
    Compute the D matrix (or its determinant) for a pair of qubits.
    
    Args:
        psi: 4-qubit state vector (16 elements) or 4-qubit tensor (2,2,2,2)
        i: First qubit index (0, 1, 2, or 3)
        j: Second qubit index (0, 1, 2, or 3)
        return_det: If True, return det(D). If False, return the 3x3 D matrix.
    
    Returns:
        If return_det=True: scalar determinant of D matrix
        If return_det=False: 3x3 numpy array D matrix
    
    Qubit labeling convention:
        0 = x, 1 = y, 2 = z, 3 = t
    
    Examples:
        # Compute D_{01} determinant (qubits 0 and 1, aka x-y pair)
        det_D01 = compute_D(psi, 0, 1)
        
        # Get the D_{23} matrix (qubits 2 and 3, aka z-t pair)
        D23_matrix = compute_D(psi, 2, 3, return_det=False)
    """
    # Ensure psi is flattened to 16 elements
    if isinstance(psi, np.ndarray) and psi.shape == (2, 2, 2, 2):
        a = psi.flatten()
    else:
        a = np.asarray(psi).flatten()
    
    assert a.shape == (16,), f"State must have 16 elements, got shape {a.shape}"
    assert i != j, f"Qubit indices must be different, got i={i}, j={j}"
    assert 0 <= i < 4 and 0 <= j < 4, f"Qubit indices must be in [0, 3], got i={i}, j={j}"
    
    # Create mapping: (i, j) sorted -> function name
    # Sort the pair to match function naming convention
    pair = tuple(sorted([i, j]))
    
    # Map qubit indices to letter names
    qubit_names = {0: 'x', 1: 'y', 2: 'z', 3: 't'}
    pair_name = ''.join([qubit_names[pair[0]], qubit_names[pair[1]]])
    
    # Dispatch to appropriate function
    dispatch_table = {
        'xy': compute_Dxy,
        'xz': compute_Dxz,
        'xt': compute_Dxt,
        'yz': compute_Dyz,
        'yt': compute_Dyt,
        'zt': compute_Dzt,
    }
    
    if pair_name not in dispatch_table:
        raise ValueError(f"Invalid qubit pair: ({i}, {j})")
    
    D_matrix = dispatch_table[pair_name](a)
    
    if return_det:
        return np.linalg.det(D_matrix)
    else:
        return D_matrix
